# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the Kenya Kilifi FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a Croissant schema hosted at:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

This notebook demonstrates how to access, process, and analyze the dataset using standard methods, referencing all schema elements by their unique `@id` fields.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant matplotlib

## 1. Data Loading

Load metadata and records from the dataset using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

import sys
import tempfile
import urllib.request

PATCH_URL = 'https://raw.githubusercontent.com/senscience/notebooks/main/tools/patch_fair2.py'
patch_folder = tempfile.mkdtemp()
urllib.request.urlretrieve(PATCH_URL, f'{patch_folder}/patch_fair2.py')
sys.path.insert(0, patch_folder)
from patch_fair2 import load_dataset

# Load the dataset metadata and initialize the mlcroissant Dataset object
dataset = load_dataset(croissant_url)

# Access metadata (note: not subscripting or iterating, just use the .metadata object)
metadata = dataset.metadata

# Display the dataset's basic metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description[:500]}...")  # Show first 500 characters

## 2. Data Overview
The dataset has one record set: one row per participant, with demographics and three screening instruments:

- **GAD-7** (Generalized Anxiety Disorder, 7 items, total 0–21),
- **PHQ-9** (Patient Health Questionnaire, depression, 9 items, total 0–27),
- **PSQ**: five probes of psychotic-like experiences (hypomania, thought insertion, paranoia, strange experiences, hallucinations), each with a follow-up question.

In [ ]:
from IPython.display import display

record_set = dataset.metadata.record_sets[0]

GAD7 = ['nervous', 'cant_stop_worrying', 'worrying_too_much', 'trouble_relaxing', 'restlessness', 'irritable', 'afraid']
PHQ9 = ['little_interest', 'hopelessness', 'sleeping_trouble', 'feeling_tired', 'poor_appetite',
        'feeling_bad_about_yourself', 'concentration', 'moving_and_speaking_slowly', 'thoughts_of_death']
PSQ_PROBES = ['hypermonia', 'thought_insertion', 'paranoia_probe', 'strange_experiences', 'hallucinations']
PSQ = PSQ_PROBES + [f'if_yes_{probe}' for probe in PSQ_PROBES]

def column_name(name):
    return name.replace('+', '_').replace(' ', '_')

def instrument(name):
    name = column_name(name)
    if name in GAD7 or name == 'gad7_total_score':
        return 'GAD-7 (anxiety)'
    if name in PHQ9 or name == 'phq9_total_score':
        return 'PHQ-9 (depression)'
    if name in PSQ or name == 'psq_total_score':
        return 'PSQ (psychotic-like experiences)'
    return 'Participant'

fields = pd.DataFrame([{'Instrument': instrument(field.name), 'Field': column_name(field.name),
                        'Description': field.description} for field in record_set.fields])
display(fields.groupby('Instrument').size().rename('Fields').to_frame()
        .style.set_caption(f'Record set {record_set.name}: fields by instrument'))

## 3. Data Extraction
Load the record set as a DataFrame, with the field names as columns (`mlcroissant` names them by field `@id`, UUIDs here) and the text decoded. The interview dates are read as dates.

In [ ]:
records = pd.DataFrame(dataset.records(record_set=record_set.id))
survey = records.rename(columns={field.id: column_name(field.name) for field in record_set.fields})
survey = survey.apply(lambda column: column.map(lambda v: v.decode('utf-8') if isinstance(v, bytes) else v))
survey['interview_date'] = pd.to_datetime(survey['date_of_interview'], format='%d/%m/%Y', errors='coerce')

display(pd.DataFrame({'Value': {'Participants': f'{len(survey):,}', 'Fields': f'{survey.shape[1] - 1}'}})
        .style.set_caption('Loaded data'))

## 4. Exploratory Data Analysis (EDA)
### 4.1 Participants

In [ ]:
age = survey['age']
profile = {
    'Participants': f'{len(survey):,}',
    **{f'{gender}': f'{count:,} ({count / len(survey):.0%})' for gender, count in survey['gender'].value_counts().items()},
    'Age, median (interquartile range)': f'{age.median():.0f} ({age.quantile(0.25):.0f}–{age.quantile(0.75):.0f})',
    'Age, range': f'{age.min()}–{age.max()}',
    'Heads of household': f"{(survey['household_head'] == 'Yes').mean():.0%}",
    'Education not recorded': f"{survey['level_of_education'].isna().sum():,}",
}
display(pd.DataFrame({'Value': profile}).style.set_properties(**{'text-align': 'right'}).set_caption('Participants'))

characteristics = pd.concat({
    column.replace('_', ' ').capitalize(): survey[column].fillna('Not recorded').value_counts()
    for column in ['village', 'level_of_education', 'marital_status']
}).rename('Participants').to_frame()
characteristics['Share (%)'] = 100 * characteristics['Participants'] / len(survey)
display(characteristics.style.format({'Participants': '{:,}', 'Share (%)': '{:.1f}'})
        .set_caption('Village, education and marital status'))

### 4.2 Data checks
The totals of the instruments are consistent with their items: the GAD-7 total is the sum of its seven items scored 0–3 (the answer *Not at all* is recorded as *Not at all sure* in this instrument), and the PSQ total is the number of *Yes* answers to the five probes and their follow-ups. The metadata describes the PSQ total as a Perceived Stress Questionnaire score, but its items are those of a psychosis screening questionnaire. The interviews took place in August and September 2024, apart from one date out of that period and two that cannot be read.

In [ ]:
FREQUENCY_SCORE = {'Not at all': 0, 'Not at all sure': 0, 'Several days': 1,
                   'Over half the days': 2, 'More than half the days': 2, 'Nearly every day': 3}
gad7_items = survey[GAD7].apply(lambda column: column.map(FREQUENCY_SCORE)).sum(axis=1)
psq_yes = survey[PSQ].apply(lambda column: column.eq('Yes')).sum(axis=1)
dates = survey['interview_date']
in_2024 = dates.dt.year.eq(2024)

display(pd.DataFrame({'Result': {
    'GAD-7 total = sum of its items': f"{(gad7_items == survey['gad7_total_score']).mean():.0%} of participants",
    "PSQ total = number of 'Yes' (probes and follow-ups)": f"{(psq_yes == survey['psq_total_score']).mean():.0%} of participants",
    'Interviews in 2024': f"{dates[in_2024].min():%d %b} – {dates[in_2024].max():%d %b %Y} ({in_2024.sum():,} participants)",
    'Interview dates outside 2024': f"{(dates.notna() & ~in_2024).sum():,} (earliest {dates.min():%Y})",
    'Interview dates not readable': f'{dates.isna().sum():,}',
}}).style.set_caption('Data checks'))

### 4.3 Grouping: severity bands
The GAD-7 and PHQ-9 totals are grouped in their standard severity bands. A score of 10 or more is the usual cut-off for screening positive for moderate anxiety (GAD-7) or depression (PHQ-9).

In [ ]:
GAD7_BANDS = pd.cut(survey['gad7_total_score'], [-1, 4, 9, 14, 21], labels=['Minimal (0–4)', 'Mild (5–9)', 'Moderate (10–14)', 'Severe (15–21)'])
PHQ9_BANDS = pd.cut(survey['phq9_total_score'], [-1, 4, 9, 14, 19, 27],
                    labels=['Minimal (0–4)', 'Mild (5–9)', 'Moderate (10–14)', 'Moderately severe (15–19)', 'Severe (20–27)'])
survey['GAD-7 band'], survey['PHQ-9 band'] = GAD7_BANDS, PHQ9_BANDS
survey['Anxiety (GAD-7 ≥ 10)'] = survey['gad7_total_score'] >= 10
survey['Depression (PHQ-9 ≥ 10)'] = survey['phq9_total_score'] >= 10

bands = pd.concat({'GAD-7': GAD7_BANDS.value_counts(sort=False), 'PHQ-9': PHQ9_BANDS.value_counts(sort=False)}).rename('Participants').to_frame()
bands['Share (%)'] = 100 * bands['Participants'] / len(survey)
display(bands.style.format({'Participants': '{:,}', 'Share (%)': '{:.1f}'}).set_caption('Severity bands'))

SCREENS = ['Anxiety (GAD-7 ≥ 10)', 'Depression (PHQ-9 ≥ 10)']
survey['Both'] = survey[SCREENS].all(axis=1)
by_gender = survey.groupby('gender')[SCREENS + ['Both']].mean().mul(100)
by_gender.loc['All participants'] = survey[SCREENS + ['Both']].mean().mul(100)
by_gender.index.name = 'Gender'
display(by_gender.style.format('{:.1f}').set_caption('Screening positive (%), by gender'))

### 4.4 Normalizing: screening positive by village and age
The villages and age groups have very different sizes: comparing the share of their participants who screen positive, rather than counts, makes them comparable. The two smallest villages (fewer than 50 participants) are shown for completeness, but their shares are uncertain.

In [ ]:
AGE_GROUPS = pd.cut(survey['age'], [17, 24, 34, 44, 54, 64, 120], labels=['18–24', '25–34', '35–44', '45–54', '55–64', '65+'])
survey['Age group'] = AGE_GROUPS

def screening_table(by):
    table = survey.groupby(by, observed=True)[SCREENS].mean().mul(100)
    table.insert(0, 'Participants', survey.groupby(by, observed=True).size())
    return table

by_village = screening_table('village').sort_values('Participants', ascending=False)
by_village.index.name = 'Village'
by_age = screening_table('Age group')
for caption, table in [('Screening positive (%), by village', by_village), ('Screening positive (%), by age group', by_age)]:
    display(table.style.format({'Participants': '{:,}', **{screen: '{:.1f}' for screen in SCREENS}})
            .background_gradient(cmap='Blues', subset=SCREENS, vmin=0, vmax=50).set_caption(caption))

## 5. Visualization
- **Anxiety and depression together**: the number of participants at each pair of GAD-7 and PHQ-9 scores, with the screening cut-offs.
- **Severity bands by gender** for both instruments.
- **Screening positive by age group.**
- **Symptoms**: the share of participants reporting each symptom on more than half the days or nearly every day.

In [ ]:
BAND_COLORS = ['#cde2fb', '#86b6ef', '#3987e5', '#1c5cab', '#0d366b']
SCREEN_COLORS = {'Anxiety (GAD-7 ≥ 10)': '#2a78d6', 'Depression (PHQ-9 ≥ 10)': '#eb6834'}
LABELS = {item: item.replace('_', ' ').replace('cant', "can't").capitalize() for item in GAD7 + PHQ9}
LABELS['hopelessness'] = 'Feeling down or hopeless'

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Anxiety and depression together
ax = axes[0, 0]
joint = pd.crosstab(survey['phq9_total_score'], survey['gad7_total_score']).reindex(index=range(28), columns=range(22), fill_value=0)
image = ax.imshow(np.log10(joint.where(joint > 0)), origin='lower', cmap='Blues', aspect='auto')
ax.axvline(9.5, color='#eb6834', linewidth=1.5, linestyle='--')
ax.axhline(9.5, color='#eb6834', linewidth=1.5, linestyle='--')
ax.set(xlabel='GAD-7 total (anxiety)', ylabel='PHQ-9 total (depression)',
       title=f"Anxiety and depression scores (Spearman ρ = {survey['gad7_total_score'].corr(survey['phq9_total_score'], method='spearman'):.2f})")
fig.colorbar(image, ax=ax, label='log₁₀(participants)')

# Severity bands by gender
ax = axes[0, 1]
rows = []
for instrument_name, band in [('GAD-7', 'GAD-7 band'), ('PHQ-9', 'PHQ-9 band')]:
    shares = pd.crosstab(survey['gender'], survey[band], normalize='index') * 100
    for gender, row in shares.iterrows():
        rows.append((f'{instrument_name} · {gender}', row))
labels = [label for label, _ in rows][::-1]
for (label, row), y in zip(rows[::-1], range(len(rows))):
    left = 0
    for (band_name, value), color in zip(row.items(), BAND_COLORS):
        ax.barh(y, value, left=left, color=color, edgecolor='white', linewidth=2)
        if value >= 5:
            ax.text(left + value / 2, y, f'{value:.0f}%', ha='center', va='center', fontsize=9,
                    color='#0b0b0b' if color in BAND_COLORS[:2] else 'white')
        left += value
ax.set_yticks(range(len(rows)), labels)
ax.set(xlim=(0, 100), xlabel='Participants (%)')
ax.set_title('Severity bands by gender', loc='left')
handles = [plt.Rectangle((0, 0), 1, 1, color=color) for color in BAND_COLORS]
ax.legend(handles, ['Minimal', 'Mild', 'Moderate', 'Moderately severe', 'Severe'], ncols=5, fontsize=8,
          loc='upper center', bbox_to_anchor=(0.5, -0.1), frameon=False)

# Screening positive by age group
ax = axes[1, 0]
x = np.arange(len(by_age))
for offset, screen in zip([-0.2, 0.2], SCREENS):
    bars = ax.bar(x + offset, by_age[screen], width=0.4, color=SCREEN_COLORS[screen], edgecolor='white', linewidth=2, label=screen)
    ax.bar_label(bars, fmt='%.0f%%', fontsize=8, padding=2)
ax.set_xticks(x, [f'{group}\n(n={n:,})' for group, n in by_age['Participants'].items()])
ax.set(ylabel='Screening positive (%)', title='Screening positive by age group')
ax.legend(fontsize=9)
ax.margins(y=0.12)

# Symptoms reported on more than half the days
ax = axes[1, 1]
frequent = {'Over half the days', 'More than half the days', 'Nearly every day'}
symptoms = pd.Series({item: survey[item].isin(frequent).mean() * 100 for item in GAD7 + PHQ9}).sort_values()
colors = [SCREEN_COLORS['Anxiety (GAD-7 ≥ 10)'] if item in GAD7 else SCREEN_COLORS['Depression (PHQ-9 ≥ 10)'] for item in symptoms.index]
bars = ax.barh([LABELS[item] for item in symptoms.index], symptoms.values, color=colors, edgecolor='white', linewidth=2)
ax.bar_label(bars, fmt='%.0f%%', fontsize=8, padding=2)
ax.set(xlabel='Participants reporting it on more than half the days (%)', title='Symptoms (GAD-7 items in blue, PHQ-9 in orange)')

plt.tight_layout()
plt.show()

## 6. Conclusion
We loaded the Kilifi mental health survey with the `mlcroissant` library and explored the screening results of its 4,188 adult participants:

- **Screening**: about one participant in eight screens positive for moderate anxiety (GAD-7 ≥ 10) and one in four for moderate depression (PHQ-9 ≥ 10); severe scores are rare.
- **Anxiety and depression go together**: the two scores are strongly correlated, and most participants who screen positive for anxiety also screen positive for depression.
- **Groups**: women screen positive slightly more often than men. Among the four large villages, Kisumu ndogo stands out, with about twice the share of the others for both screens; the 35–44 age group has the highest shares. The two smallest villages are too small to compare.
- **Data checks**: the instrument totals match their items; the PSQ total counts psychotic-like experiences rather than perceived stress, as its description says; one interview date is out of range, two cannot be read, and the education level is missing for about a quarter of the participants.

Further exploration can model the screening results by demographic factors, study the PSQ items, or compare the villages adjusting for age and gender.